In [ ]:
# This file generates predictions and diagrams for the de Almeida feeds. This data is used for figure plotting, in experimental_ternary_figures.ipynb

using CSV, DataFrames, Statistics, Plots, LinearAlgebra

include("../NRTL_model.jl")  # For activity_coefficients
include("../calculate_phase_split.jl")   # For solve_lle

# lit data----
# Table 4: Cloud points at T = 303.2 K (closest to 298 K = 25°C)
# Organic phase (EA-rich) mass fractions: w1=water, w2=furfural, w3=EA

println("=== LITERATURE VALIDATION (MOLE FRACTION BASIS) ===")
println("Source: Table 4 - Cloud points at T = 303.2 K")
println("Converting literature data to mole fractions\n")

# lit data at 303.2 K (30°C)---organic phase (mass fractions)
w_lit_organic = [
    [0.0346, 0.0000, 0.9654],
    [0.0404, 0.1157, 0.8439],
    [0.0480, 0.2378, 0.7142],
    [0.0507, 0.3586, 0.5907],
    [0.0566, 0.4729, 0.4705],
    [0.0574, 0.5716, 0.3710],
    [0.0603, 0.7538, 0.1859],
    [0.0602, 0.8052, 0.1346],
    [0.0593, 0.9407, 0.0000]
]


MW = [18.015, 96.084, 88.106]  # Water, Furfural, EA

# comp densities
ρ_water = 1.00
ρ_furfural = 1.16
ρ_EA = 0.902

# helpers---
function mass_to_mole(w)
    n = w ./ MW
    return n ./ sum(n)
end

function mole_to_mass(x)
    w = x .* MW
    return w ./ sum(w)
end

function mixture_density(w)
    return 1.0 / (w[1]/ρ_water + w[2]/ρ_furfural + w[3]/ρ_EA)
end

# convert literature data to mole fractions
x_lit_organic = [mass_to_mole(w) for w in w_lit_organic]

println("Literature Organic Phase Compositions (mole fractions):")
println("Point | x_water | x_furf | x_EA")
println("-" ^ 50)
for (i, x) in enumerate(x_lit_organic)
    println("$i     | $(round(x[1], digits=4)) | $(round(x[2], digits=4)) | $(round(x[3], digits=4))")
end

τ_file = "../fitted_tau_matrix_0.csv"
τ_fitted = CSV.read(τ_file, DataFrame) |> Matrix{Float64}


# predict equilibrium phases---

x_pred_aqueous = []
x_pred_organic = []
convergence_flags = []
composition_errors = []

for (i, x_org_lit) in enumerate(x_lit_organic)
    println("="^70)
    println("Literature Point $i:")
    println("  Organic phase (lit): x = [$(round(x_org_lit[1], digits=4)), " *
           "$(round(x_org_lit[2], digits=4)), $(round(x_org_lit[3], digits=4))]")
    
    # try different overall feed compositions to find one that 
    # produces an organic phase close to literature
    best_match = nothing
    best_error = Inf
    best_feed = nothing
    
    # try feeds spanning from water-rich to organic-rich
    for α in 0.1:0.1:0.9
        # create feed between pure water and literature organic composition
        z_feed_mole = α .* x_org_lit .+ (1 - α) .* [1.0, 0.0, 0.0]
        z_feed_mole = z_feed_mole ./ sum(z_feed_mole)
        
        try
            phase_aq, phase_org = solve_lle(z_feed_mole, τ_fitted)
            
            x_aq_pred = phase_aq
            x_org_pred = phase_org
            
            # check which phase matches literature organic phase better
            error_as_org = sqrt(sum((x_org_pred .- x_org_lit).^2))
            error_as_aq = sqrt(sum((x_aq_pred .- x_org_lit).^2))
            
            if error_as_org < error_as_aq && error_as_org < best_error
                best_error = error_as_org
                best_match = (x_aq_pred, x_org_pred)
                best_feed = z_feed_mole
            elseif error_as_aq < error_as_org && error_as_aq < best_error
                # Literature point matched aqueous phase instead (swap)
                best_error = error_as_aq
                best_match = (x_org_pred, x_aq_pred)
                best_feed = z_feed_mole
            end
        catch e
            continue
        end
    end
    
    if best_match !== nothing && best_error < 0.2  # Reasonable match threshold
        x_aq, x_org = best_match
        push!(x_pred_aqueous, x_aq)
        push!(x_pred_organic, x_org)
        push!(convergence_flags, true)
        push!(composition_errors, best_error)
        
        println("Found matching equilibrium (error = $(round(best_error, digits=4)))")
        println("Predicted organic: x = [$(round(x_org[1], digits=4)), " *
               "$(round(x_org[2], digits=4)), $(round(x_org[3], digits=4))]")
        println("Predicted aqueous: x = [$(round(x_aq[1], digits=4)), " *
               "$(round(x_aq[2], digits=4)), $(round(x_aq[3], digits=4))]")
        
        # Show component-wise errors
        err = x_org .- x_org_lit
        println("Component errors: ΔX_W=$(round(err[1], digits=4)), " *
               "ΔX_F=$(round(err[2], digits=4)), ΔX_EA=$(round(err[3], digits=4))")
    else
        println("Could not find matching equilibrium (best error: $(round(best_error, digits=4)))")
        push!(x_pred_aqueous, [NaN, NaN, NaN])
        push!(x_pred_organic, [NaN, NaN, NaN])
        push!(convergence_flags, false)
        push!(composition_errors, NaN)
    end
    println()
end

# quantitative validation
println("\n" * "="^70)
println("QUANTITATIVE VALIDATION METRICS\n")

valid_indices = findall(convergence_flags)
n_valid = length(valid_indices)
n_total = length(x_lit_organic)

println("Successful predictions: $n_valid / $n_total ($(round(n_valid/n_total*100, digits=1))%)")

if n_valid > 0
    # Component-wise errors (mole fraction basis)
    errors_water = [x_pred_organic[i][1] - x_lit_organic[i][1] for i in valid_indices]
    errors_furf = [x_pred_organic[i][2] - x_lit_organic[i][2] for i in valid_indices]
    errors_ea = [x_pred_organic[i][3] - x_lit_organic[i][3] for i in valid_indices]
    
    println("\nOrganic Phase Composition Errors (predicted - literature, mole fractions):")
    println("  Water:")
    println("    MAE  = $(round(mean(abs.(errors_water)), digits=4))")
    println("    RMSE = $(round(sqrt(mean(errors_water.^2)), digits=4))")
    println("    Bias = $(round(mean(errors_water), digits=4))")
    
    println("  Furfural:")
    println("    MAE  = $(round(mean(abs.(errors_furf)), digits=4))")
    println("    RMSE = $(round(sqrt(mean(errors_furf.^2)), digits=4))")
    println("    Bias = $(round(mean(errors_furf), digits=4))")
    
    println("  EA:")
    println("    MAE  = $(round(mean(abs.(errors_ea)), digits=4))")
    println("    RMSE = $(round(sqrt(mean(errors_ea.^2)), digits=4))")
    println("    Bias = $(round(mean(errors_ea), digits=4))")
    
    # Overall metrics
    all_errors = vcat(errors_water, errors_furf, errors_ea)
    println("\nOverall:")
    println("  MAE  = $(round(mean(abs.(all_errors)), digits=4))")
    println("  RMSE = $(round(sqrt(mean(all_errors.^2)), digits=4))")
    
    # R² by component
    println("\nR² by Component (in mole fraction space):")
    for (comp_idx, comp_name) in enumerate(["Water", "Furfural", "EA"])
        x_lit_comp = [x_lit_organic[i][comp_idx] for i in valid_indices]
        x_pred_comp = [x_pred_organic[i][comp_idx] for i in valid_indices]
        
        SS_res = sum((x_pred_comp .- x_lit_comp).^2)
        SS_tot = sum((x_lit_comp .- mean(x_lit_comp)).^2)
        R2 = 1 - SS_res / SS_tot
        
        println("  $comp_name: R² = $(round(R2, digits=4))")
    end
    
    # Overall R² (all components together)
    all_lit = vcat([x_lit_organic[i] for i in valid_indices]...)
    all_pred = vcat([x_pred_organic[i] for i in valid_indices]...)
    SS_res_total = sum((all_pred .- all_lit).^2)
    SS_tot_total = sum((all_lit .- mean(all_lit)).^2)
    R2_total = 1 - SS_res_total / SS_tot_total
    println("  Overall: R² = $(round(R2_total, digits=4))")
    
    # Quality assessment
    println("\nMODEL QUALITY ASSESSMENT")
    overall_mae = mean(abs.(all_errors))
    if overall_mae < 0.03
        println("GREAT: MAE < 0.03 in mole fraction")
    elseif overall_mae < 0.05
        println("GOOD: MAE < 0.05 in mole fraction")
    elseif overall_mae < 0.10
        println("FAIR: MAE < 0.10 in mole fraction")
    else
        println("POOR: MAE > 0.10 in mole fraction")
        println("Model parameters may need refitting or temperature correction")
    end
    
else
    println("No valid predictions - model may be fundamentally wrong")
end

gr()

# Plot 1: Ternary diagram
p1 = plot(
    aspect_ratio=:equal,
    legend=:topright,
    size=(900, 900),
    title="Literature Validation (T=303.2K) - Mole Fractions",
    xlabel="",
    ylabel="",
    grid=false,
    showaxis=false,
    margin=5Plots.mm
)

# Draw triangle
triangle_x = [0, 1, 0.5, 0]
triangle_y = [0, 0, sqrt(3)/2, 0]
plot!(p1, triangle_x, triangle_y, color=:black, linewidth=2, label="")

# Labels
annotate!(p1, 0.25, sqrt(3)/2 + 0.05, text("Water", :center, 12, :bold))
annotate!(p1, -0.08, -0.02, text("Furfural", :center, 12, :bold))
annotate!(p1, 1.08, -0.02, text("EA", :center, 12, :bold))

# Helper function for ternary coordinates
function ternary_to_cartesian(x_water, x_furf, x_ea)
    x = 0.5 * (2*x_ea + x_furf)
    y = (sqrt(3)/2) * x_furf
    return x, y
end

# Plot literature organic phase data
for (i, x) in enumerate(x_lit_organic)
    px, py = ternary_to_cartesian(x[1], x[2], x[3])
    scatter!(p1, [px], [py], color=:blue, marker=:circle, markersize=10, 
            label=(i==1 ? "Literature (organic)" : ""))
    annotate!(p1, px+0.02, py+0.02, text("$i", 8))
end

# Plot predicted tie lines
for (i, (x_aq, x_org)) in enumerate(zip(x_pred_aqueous, x_pred_organic))
    if convergence_flags[i]
        px1, py1 = ternary_to_cartesian(x_aq[1], x_aq[2], x_aq[3])
        px2, py2 = ternary_to_cartesian(x_org[1], x_org[2], x_org[3])
        
        # Tie line
        plot!(p1, [px1, px2], [py1, py2], color=:red, linewidth=2, linestyle=:dash,
             label=(i==1 ? "Model prediction" : ""), alpha=0.7)
        
        # Points
        scatter!(p1, [px1], [py1], color=:red, marker=:square, markersize=7, label="")
        scatter!(p1, [px2], [py2], color=:red, marker=:square, markersize=7, label="")
    end
end

# Plot 2: Component parity plots
p2 = scatter(
    title="Water (Organic Phase)",
    xlabel="Literature x_water",
    ylabel="Predicted x_water",
    legend=:topleft,
    markersize=10,
    color=:blue,
    size=(500, 500)
)

if n_valid > 0
    x_lit_water = [x_lit_organic[i][1] for i in valid_indices]
    x_pred_water = [x_pred_organic[i][1] for i in valid_indices]
    scatter!(p2, x_lit_water, x_pred_water, label="Data")
    
    lim = max(maximum(x_lit_water), maximum(x_pred_water)) * 1.1
    plot!(p2, [0, lim], [0, lim], color=:black, linestyle=:dash, linewidth=2, label="y=x")
end

p3 = scatter(
    title="Furfural (Organic Phase)",
    xlabel="Literature x_furfural",
    ylabel="Predicted x_furfural",
    legend=:topleft,
    markersize=10,
    color=:green,
    size=(500, 500)
)

if n_valid > 0
    x_lit_furf = [x_lit_organic[i][2] for i in valid_indices]
    x_pred_furf = [x_pred_organic[i][2] for i in valid_indices]
    scatter!(p3, x_lit_furf, x_pred_furf, label="Data")
    
    lim = max(maximum(x_lit_furf), maximum(x_pred_furf)) * 1.05
    plot!(p3, [0, lim], [0, lim], color=:black, linestyle=:dash, linewidth=2, label="y=x")
end

p4 = scatter(
    title="EA (Organic Phase)",
    xlabel="Literature x_EA",
    ylabel="Predicted x_EA",
    legend=:topleft,
    markersize=10,
    color=:red,
    size=(500, 500)
)

if n_valid > 0
    x_lit_ea = [x_lit_organic[i][3] for i in valid_indices]
    x_pred_ea = [x_pred_organic[i][3] for i in valid_indices]
    scatter!(p4, x_lit_ea, x_pred_ea, label="Data")
    
    lim = max(maximum(x_lit_ea), maximum(x_pred_ea)) * 1.05
    plot!(p4, [0, lim], [0, lim], color=:black, linestyle=:dash, linewidth=2, label="y=x")
end

p_parity = plot(p2, p3, p4, layout=(1,3), size=(1500, 500))


# Save results to CSV
if n_valid > 0
    results_df = DataFrame(
        Point = valid_indices,
        Lit_x_water = [x_lit_organic[i][1] for i in valid_indices],
        Lit_x_furf = [x_lit_organic[i][2] for i in valid_indices],
        Lit_x_EA = [x_lit_organic[i][3] for i in valid_indices],
        Pred_x_water = [x_pred_organic[i][1] for i in valid_indices],
        Pred_x_furf = [x_pred_organic[i][2] for i in valid_indices],
        Pred_x_EA = [x_pred_organic[i][3] for i in valid_indices],
        Error_water = errors_water,
        Error_furf = errors_furf,
        Error_EA = errors_ea,
        Composition_error = [composition_errors[i] for i in valid_indices]
    )
    
    CSV.write(joinpath(output_dir, "literature_validation_mole_basis.csv"), results_df)
    println("Results saved to 'literature_validation_mole_basis.csv'")
end

display(p1)

In [ ]:
using CSV, DataFrames, Plots, LaTeXStrings

gr()

csv_file = "literature_validation/literature_validation_mole_basis_fittosalted.csv" # The previous script, ran for the salt = 2.5% tau parameter matrix!
output_dir = "literature_validation"
mkpath(output_dir)

default(
    fontfamily = "Arial",
    guidefontsize = 20,
    tickfontsize = 13,
    legendfontsize = 16,
    linewidth = 2.0,
    markersize = 10,
    left_margin = 14Plots.mm,
    bottom_margin = 12Plots.mm
)

df = CSV.read(csv_file, DataFrame)

x_lit_water = df.Lit_x_water
x_pred_water = df.Pred_x_water

x_lit_furf = df.Lit_x_furf
x_pred_furf = df.Pred_x_furf

x_lit_ea = df.Lit_x_EA
x_pred_ea = df.Pred_x_EA

all_lit_vals = vcat(x_lit_water, x_lit_furf, x_lit_ea)
all_pred_vals = vcat(x_pred_water, x_pred_furf, x_pred_ea)

lim_min = min(minimum(all_lit_vals), minimum(all_pred_vals))
lim_max = max(maximum(all_lit_vals), maximum(all_pred_vals))
pad = 0.03
lims = (max(0.0, lim_min - pad), min(1.0, lim_max + pad))

p1 = scatter(
    x_lit_water, x_pred_water,
    xlabel = L"\mathrm{Literature\ x_{water, org}}",
    ylabel = L"\mathrm{Predicted\ x_{water, org}}",
    label = "Water",
    color = :blue,
    marker = :circle,
    markersize = 10,
    markerstrokewidth = 0.5,
    legend = :topleft,
    grid = false,
    framestyle = :box,
    xlims = lims,
    ylims = lims,
    aspect_ratio = :equal,
    margin = 6Plots.mm
)

plot!(
    p1, [lims[1], lims[2]], [lims[1], lims[2]],
    color = :black,
    linestyle = :dash,
    linewidth = 2,
    label = "y = x"
)

p2 = scatter(
    x_lit_furf, x_pred_furf,
    xlabel = L"\mathrm{Literature\ x_{furfural, org}}",
    ylabel = L"\mathrm{Predicted\ x_{furfural, org}}",
    label = "Furfural",
    color = :forestgreen,
    marker = :diamond,
    markersize = 10,
    markerstrokewidth = 0.5,
    legend = :topleft,
    grid = false,
    framestyle = :box,
    xlims = lims,
    ylims = lims,
    aspect_ratio = :equal,
    margin = 6Plots.mm
)

plot!(
    p2, [lims[1], lims[2]], [lims[1], lims[2]],
    color = :black,
    linestyle = :dash,
    linewidth = 2,
    label = "y = x"
)

p3 = scatter(
    x_lit_ea, x_pred_ea,
    xlabel = L"\mathrm{Literature\ x_{EA, org}}",
    ylabel = L"\mathrm{Predicted\ x_{EA, org}}",
    label = "EA",
    color = :red,
    marker = :utriangle,
    markersize = 10,
    markerstrokewidth = 0.5,
    legend = :topleft,
    grid = false,
    framestyle = :box,
    xlims = lims,
    ylims = lims,
    aspect_ratio = :equal,
    margin = 6Plots.mm
)

plot!(
    p3, [lims[1], lims[2]], [lims[1], lims[2]],
    color = :black,
    linestyle = :dash,
    linewidth = 2,
    label = "y = x"
)

p_final = plot(
    p1, p2, p3,
    layout = (1, 3),
    size = (2000, 600)
)

savefig(p_final, joinpath(output_dir, "parity_plots_mole_basis_fittosalted.png"))
savefig(p_final, joinpath(output_dir, "parity_plots_mole_basis_fittosalted.pdf"))

"/Users/laracapellino/Documents/GitHub/fullscale_model_SALLE_jl/fullscale_predictive_model_jl_massconv/Experimental_Validation/literature_validation/parity_plots_mole_basis_fittosalted.pdf"

In [1]:
using CSV, DataFrames, Plots, LaTeXStrings

gr()

csv_file = "literature_validation/literature_validation_mole_basis.csv"
output_dir = "literature_validation"
mkpath(output_dir)

default(
    fontfamily = "Arial",
    guidefontsize = 20,
    tickfontsize = 13,
    legendfontsize = 16,
    linewidth = 2.0,
    markersize = 10,
    left_margin = 14Plots.mm,
    bottom_margin = 12Plots.mm
)

df = CSV.read(csv_file, DataFrame)

x_lit_water = df.Lit_x_water
x_pred_water = df.Pred_x_water

x_lit_furf = df.Lit_x_furf
x_pred_furf = df.Pred_x_furf

x_lit_ea = df.Lit_x_EA
x_pred_ea = df.Pred_x_EA

all_lit_vals = vcat(x_lit_water, x_lit_furf, x_lit_ea)
all_pred_vals = vcat(x_pred_water, x_pred_furf, x_pred_ea)

lim_min = min(minimum(all_lit_vals), minimum(all_pred_vals))
lim_max = max(maximum(all_lit_vals), maximum(all_pred_vals))
pad = 0.03
lims = (max(0.0, lim_min - pad), min(1.0, lim_max + pad))

p1 = scatter(
    x_lit_water, x_pred_water,
    xlabel = L"\mathrm{Literature\ x_{water, org}}",
    ylabel = L"\mathrm{Predicted\ x_{water, org}}",
    label = "Water",
    color = :blue,
    marker = :circle,
    markersize = 10,
    markerstrokewidth = 0.5,
    legend = :topleft,
    grid = false,
    framestyle = :box,
    xlims = lims,
    ylims = lims,
    aspect_ratio = :equal,
    margin = 6Plots.mm
)

plot!(
    p1, [lims[1], lims[2]], [lims[1], lims[2]],
    color = :black,
    linestyle = :dash,
    linewidth = 2,
    label = "y = x"
)

p2 = scatter(
    x_lit_furf, x_pred_furf,
    xlabel = L"\mathrm{Literature\ x_{furfural, org}}",
    ylabel = L"\mathrm{Predicted\ x_{furfural, org}}",
    label = "Furfural",
    color = :forestgreen,
    marker = :diamond,
    markersize = 10,
    markerstrokewidth = 0.5,
    legend = :topleft,
    grid = false,
    framestyle = :box,
    xlims = lims,
    ylims = lims,
    aspect_ratio = :equal,
    margin = 6Plots.mm
)

plot!(
    p2, [lims[1], lims[2]], [lims[1], lims[2]],
    color = :black,
    linestyle = :dash,
    linewidth = 2,
    label = "y = x"
)

p3 = scatter(
    x_lit_ea, x_pred_ea,
    xlabel = L"\mathrm{Literature\ x_{EA, org}}",
    ylabel = L"\mathrm{Predicted\ x_{EA, org}}",
    label = "EA",
    color = :red,
    marker = :utriangle,
    markersize = 10,
    markerstrokewidth = 0.5,
    legend = :topleft,
    grid = false,
    framestyle = :box,
    xlims = lims,
    ylims = lims,
    aspect_ratio = :equal,
    margin = 6Plots.mm
)

plot!(
    p3, [lims[1], lims[2]], [lims[1], lims[2]],
    color = :black,
    linestyle = :dash,
    linewidth = 2,
    label = "y = x"
)

p_final = plot(
    p1, p2, p3,
    layout = (1, 3),
    size = (2000, 600)
)

savefig(p_final, joinpath(output_dir, "parity_plots_mole_basis.png"))
savefig(p_final, joinpath(output_dir, "parity_plots_mole_basis.pdf"))

"/Users/laracapellino/Documents/GitHub/fullscale_model_SALLE_jl/fullscale_predictive_model_jl_massconv/Experimental_Validation/literature_validation/parity_plots_mole_basis.pdf"